# BuildCompiler Transformation Quickstart

This notebook runs a local level-1 assembly from `tests/test_files/abstract_design.xml`, then uses the assembly product as the input for chemical transformation.

## 1. Imports and Local Paths

Run this notebook from the repository checkout after installing BuildCompiler with `python -m pip install -e .`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import sbol2

from buildcompiler.abstract_translator import extract_toplevel_definition
from buildcompiler.adapters.pudu import plating_to_pudu_json, transformations_to_pudu_json, write_assembly_pudu_input_json
from buildcompiler.buildcompiler import BuildCompiler


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "tests" / "test_files").exists():
            return path
    raise RuntimeError("Could not find BuildCompiler repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
TEST_FILES = REPO_ROOT / "tests" / "test_files"
RESULTS_DIR = REPO_ROOT / "notebooks" / "results" / "buildcompiler_transformation_quickstart"
PUDU_REPO = REPO_ROOT.parent / "PUDU"
PUDU_SRC = PUDU_REPO / "src"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Results:", RESULTS_DIR)
print("PUDU source:", PUDU_SRC if PUDU_SRC.exists() else "not found")

## 2. Load Offline SBOL Collections

The compiler indexes local plasmids, backbones, and enzyme implementations from these fixture collections.

In [ ]:
collection_paths = [
    TEST_FILES / "CIDARMoCloParts_collection.xml",
    TEST_FILES / "CIDARMoCloPlasmidsKit_collection.xml",
    TEST_FILES / "Enzyme_Implementations_collection.xml",
    TEST_FILES / "impl_test_collection.xml",
]

collection_docs = []
for path in collection_paths:
    doc = sbol2.Document()
    doc.read(str(path))
    collection_docs.append(doc)
    print(path.name, "components=", len(doc.componentDefinitions), "implementations=", len(doc.implementations))

## 3. Build Level-1 Assembly Products

`abstract_design.xml` is resolved against the local collections and assembled into a level-1 plasmid product.

In [ ]:
design_doc = sbol2.Document()
design_doc.read(str(TEST_FILES / "abstract_design.xml"))
design = extract_toplevel_definition(design_doc)

compiler = BuildCompiler.from_local_documents(collection_docs, design_doc=design_doc)

assembly_doc = sbol2.Document()
assembly_routes, assembly_doc = compiler.assembly_lvl1(
    [design],
    final_doc=assembly_doc,
    product_name="transformation_lvl1",
)
assembly_products = assembly_routes[design.identity]

lvl1_sbol_path = RESULTS_DIR / "transformation_lvl1_products.xml"
lvl1_pudu_path = RESULTS_DIR / "assembly_lvl1_pudu_input.json"
lvl1_sbol_path.write_text(assembly_doc.writeString(), encoding="utf-8")
write_assembly_pudu_input_json(compiler.last_assembly_pudu_json, lvl1_pudu_path)

print("Design:", design.displayId)
print("Assembly products:", [product.plasmid_definition.displayId for product in assembly_products])
print("Level-1 SBOL:", lvl1_sbol_path)
print("Level-1 assembly PUDU input:", lvl1_pudu_path)

## 4. Transform the Level-1 Product

The transformation stage consumes the structured level-1 assembly output directly and writes PUDU-compatible transformation inputs.

In [ ]:
CHASSIS_NAME = "E_coli_DH5alpha"
PUDU_CHASSIS_URI = "https://sbolcanvas.org/DH5alpha/1"

transformation_result = compiler.transformation(
    assembly_products,
    chassis_name=CHASSIS_NAME,
    transformation_doc=assembly_doc,
)

transformation_sbol_path = RESULTS_DIR / "transformation_products.xml"
transformation_summary_path = RESULTS_DIR / "transformation_summary.json"
transformation_pudu_path = RESULTS_DIR / "transformation_lvl1_pudu_input.json"

transformation_sbol_path.write_text(assembly_doc.writeString(), encoding="utf-8")
transformation_summary_path.write_text(json.dumps(transformation_result, indent=2), encoding="utf-8")

transformation_pudu = transformations_to_pudu_json(
    strain_identities=[artifact["transformed_strain_module"] for artifact in transformation_result["sbol_artifacts"]],
    chassis_identities=[PUDU_CHASSIS_URI for _ in transformation_result["sbol_artifacts"]],
    plasmid_sets=[[product.plasmid_definition.identity] for product in assembly_products],
)
transformation_pudu_path.write_text(json.dumps(transformation_pudu, indent=2), encoding="utf-8")

print("Transformation inputs:", transformation_result["inputs"])
print("Transformation SBOL:", transformation_sbol_path)
print("Transformation summary:", transformation_summary_path)
print("Transformation PUDU spec:", transformation_pudu_path)

## 5. Inspect the Generated Records

In [ ]:
print(json.dumps(transformation_pudu, indent=2))
print("Transformation SBOL artifacts:")
for artifact in transformation_result["sbol_artifacts"]:
    print(json.dumps(artifact, indent=2))

## 6. Run the PUDU Assembly to Transformation to Plating Chain

PUDU uses `opentrons_simulate` as the handoff between stages: assembly simulation writes `transformation_input.json`, transformation simulation writes `plating_input.json`, and plating simulation writes the final plating layout files.

In [ ]:
if not PUDU_SRC.exists():
    raise RuntimeError(f"PUDU source tree not found at {PUDU_SRC}")
if str(PUDU_SRC) not in sys.path:
    sys.path.insert(0, str(PUDU_SRC))

from pudu.generate_protocol import detect_protocol_type, generate_protocol


def run_opentrons_simulation(protocol_path: Path) -> Path:
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH")
    env["PYTHONPATH"] = str(PUDU_SRC) if not existing_pythonpath else f"{PUDU_SRC}{os.pathsep}{existing_pythonpath}"
    log_path = protocol_path.with_suffix(".simulate.log")
    result = subprocess.run(
        ["opentrons_simulate", protocol_path.name],
        cwd=RESULTS_DIR,
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
    log_path.write_text(result.stdout + result.stderr, encoding="utf-8")
    return log_path


assembly_pudu = json.loads(lvl1_pudu_path.read_text(encoding="utf-8"))
protocol_type, assembly_subtype = detect_protocol_type(assembly_pudu)
assert (protocol_type, assembly_subtype) == ("assembly", "SBOL")
pudu_assembly_protocol_path = RESULTS_DIR / "pudu_assembly_protocol.py"
pudu_assembly_protocol_path.write_text(
    generate_protocol(
        protocol_data=assembly_pudu,
        protocol_type="assembly",
        assembly_subtype="SBOL",
        metadata={
            "protocolName": "BuildCompiler Level-1 Assembly",
            "author": "BuildCompiler",
            "description": "PUDU assembly generated from BuildCompiler level-1 input.",
        },
    ),
    encoding="utf-8",
)
assembly_log_path = run_opentrons_simulation(pudu_assembly_protocol_path)
transformation_locations_path = RESULTS_DIR / "transformation_input.json"
plasmid_locations = json.loads(transformation_locations_path.read_text(encoding="utf-8"))

protocol_type, assembly_subtype = detect_protocol_type(transformation_pudu)
assert (protocol_type, assembly_subtype) == ("transformation", None)

pudu_protocol_path = RESULTS_DIR / "pudu_transformation_protocol.py"
pudu_protocol_code = generate_protocol(
    protocol_data=transformation_pudu,
    protocol_type="transformation",
    plasmid_locations=plasmid_locations,
    metadata={
        "protocolName": "BuildCompiler Level-1 Transformation",
        "author": "BuildCompiler",
        "description": "PUDU heat-shock transformation generated from BuildCompiler level-1 output.",
    },
)
pudu_protocol_path.write_text(pudu_protocol_code, encoding="utf-8")
transformation_log_path = run_opentrons_simulation(pudu_protocol_path)
plating_input_path = RESULTS_DIR / "plating_input.json"
plating_input = json.loads(plating_input_path.read_text(encoding="utf-8"))
plating_input = plating_to_pudu_json(bacterium_locations=plating_input["bacterium_locations"])
plating_input_path.write_text(json.dumps(plating_input, indent=2), encoding="utf-8")

protocol_type, assembly_subtype = detect_protocol_type(plating_input)
assert (protocol_type, assembly_subtype) == ("plating", None)
pudu_plating_protocol_path = RESULTS_DIR / "pudu_plating_protocol.py"
pudu_plating_protocol_path.write_text(
    generate_protocol(
        protocol_data=plating_input,
        protocol_type="plating",
        metadata={
            "protocolName": "BuildCompiler Plating",
            "author": "BuildCompiler",
            "description": "PUDU plating generated from BuildCompiler transformation output.",
        },
    ),
    encoding="utf-8",
)
plating_log_path = run_opentrons_simulation(pudu_plating_protocol_path)

print("Generated PUDU assembly protocol:", pudu_assembly_protocol_path)
print("Assembly simulation log:", assembly_log_path)
print("Assembly output for transformation:", transformation_locations_path)
print("Generated PUDU protocol:", pudu_protocol_path)
print("Transformation simulation log:", transformation_log_path)
print("Transformation output for plating:", plating_input_path)
print("Generated PUDU plating protocol:", pudu_plating_protocol_path)
print("Plating simulation log:", plating_log_path)
print("Final plating layout JSON:", RESULTS_DIR / "plating_layout.json")